In [1]:
import pandas as pd

df =pd.read_csv('100_Unique_QA_Dataset.csv')
df.head()

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


In [2]:
def tokenize(text):
    text = text.lower()
    text= text.replace("?","")
    text= text.replace("!","")
    text= text.replace("'","")
    return text.split()

In [3]:
tokenize('What is the capital of France?')

['what', 'is', 'the', 'capital', 'of', 'france']

In [4]:
vocab = {'<UNK>':0}

In [10]:
def build_vocab(row):
    tokenized_question = tokenize(row['question'])
    tokenized_answer = tokenize(row['answer'])

    merged_token = tokenized_question + tokenized_answer

    for token in merged_token:
        if token not in vocab:
            vocab[token] = len(vocab)

In [14]:
df.apply(build_vocab, axis=1)

0     None
1     None
2     None
3     None
4     None
      ... 
85    None
86    None
87    None
88    None
89    None
Length: 90, dtype: object

In [15]:
len(vocab)

324

In [16]:
def text_to_indices(text,vocab):
    indexed_text= []

    for token in tokenize(text):
        if token in vocab:
            indexed_text.append(vocab[token])
        else:
            indexed_text.append(0)
    return indexed_text


In [17]:
import torch
from torch.utils.data import  DataLoader, Dataset

In [18]:
class CustomDataset(Dataset):

    def __init__(self, df, vocab):
        self.data = df
        self.vocab = vocab

    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        question = text_to_indices(self.data.iloc[idx]['question'],self.vocab)
        answer = text_to_indices(self.data.iloc[idx]['answer'],self.vocab)

        return torch.tensor(question), torch.tensor(answer)

In [19]:
train_dataset = CustomDataset(df, vocab)

In [20]:
train_dataloader = DataLoader(train_dataset, batch_size=1, shuffle=True)

In [21]:
for question , answer in train_dataloader:
    print(question)
    print(answer)

tensor([[  1,   2,   3,   4,   5, 135]])
tensor([[136]])
tensor([[10, 29,  3, 30, 31]])
tensor([[32]])
tensor([[ 10, 140,   3, 141, 171,   5,   3,  70, 172]])
tensor([[173]])
tensor([[ 42, 263, 264,  14, 265, 266, 158, 267]])
tensor([[268]])
tensor([[ 1,  2,  3, 92, 93, 94]])
tensor([[95]])
tensor([[ 42, 137,   2,  62,  39,   3, 322, 323]])
tensor([[6]])
tensor([[ 78,  79, 195,  81,  19,   3, 196, 197, 198]])
tensor([[199]])
tensor([[  1,   2,   3,   4,   5, 236, 237]])
tensor([[238]])
tensor([[  1,   2,   3,   4,   5, 279]])
tensor([[280]])
tensor([[ 78,  79, 288,  81,  19,  14, 289]])
tensor([[85]])
tensor([[ 1,  2,  3, 69,  5, 53]])
tensor([[260]])
tensor([[ 1,  2,  3,  4,  5, 73]])
tensor([[74]])
tensor([[ 42,  18, 118,   3, 186, 187]])
tensor([[188]])
tensor([[ 1,  2,  3,  4,  5, 53]])
tensor([[54]])
tensor([[1, 2, 3, 4, 5, 8]])
tensor([[9]])
tensor([[ 42, 250, 251, 118, 252, 253]])
tensor([[254]])
tensor([[ 10, 308,   3, 309, 310]])
tensor([[311]])
tensor([[42, 43, 44, 45, 46, 47

In [22]:
import torch.nn as nn

In [45]:
class SimpleNN(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(len(vocab), embedding_dim=50)
        self.rnn = nn.RNN(50, 64, batch_first=True)
        self.linear = nn.Linear(64, vocab_size)

    def forward(self, question):
        embedded = self.embedding(question)
        hidden,final = self.rnn(embedded)
        output = self.linear(final.squeeze(0))
        return output

In [46]:
x = nn.Embedding(324, embedding_dim=50)
y = nn.RNN(50, 64, batch_first=True)
z = nn.Linear(64, 324)

a = train_dataset[0][0].reshape(1,6)
print("shape of a:", a.shape)
b = x(a)
print("shape of b:", b.shape)
c, d = y(b)
print("shape of c:", c.shape)
print("shape of d:", d.shape)

e = z(d.squeeze(0))

print("shape of e:", e.shape)

shape of a: torch.Size([1, 6])
shape of b: torch.Size([1, 6, 50])
shape of c: torch.Size([1, 6, 64])
shape of d: torch.Size([1, 1, 64])
shape of e: torch.Size([1, 324])


In [47]:
learning_rate = 0.001
epochs = 20

In [48]:
model = SimpleNN(len(vocab))

In [49]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [54]:
# training loop

for epoch in range(epochs):

  total_loss = 0

  for question, answer in train_dataloader:

    optimizer.zero_grad()

    # forward pass
    output = model(question)
    # print(output)

    # loss -> output shape (1,324) - (1)
    loss = criterion(output, answer[0]
                     )

    # gradients
    loss.backward()

    # update
    optimizer.step()

    total_loss = total_loss + loss.item()

  print(f"Epoch: {epoch+1}, Loss: {total_loss:4f}")

Epoch: 1, Loss: 1.611511
Epoch: 2, Loss: 1.508852
Epoch: 3, Loss: 1.417919
Epoch: 4, Loss: 1.330909
Epoch: 5, Loss: 1.251021
Epoch: 6, Loss: 1.178498
Epoch: 7, Loss: 1.109513
Epoch: 8, Loss: 1.044737
Epoch: 9, Loss: 0.986160
Epoch: 10, Loss: 0.930395
Epoch: 11, Loss: 0.879093
Epoch: 12, Loss: 0.830482
Epoch: 13, Loss: 0.785925
Epoch: 14, Loss: 0.743623
Epoch: 15, Loss: 0.703655
Epoch: 16, Loss: 0.666485
Epoch: 17, Loss: 0.631250
Epoch: 18, Loss: 0.598776
Epoch: 19, Loss: 0.567593
Epoch: 20, Loss: 0.538737


In [59]:
def predict(model, question, threshold=0.7):

  # convert question to numbers
  numerical_question = text_to_indices(question, vocab)

  # tensor
  question_tensor = torch.tensor(numerical_question).unsqueeze(0)

  # send to model
  output = model(question_tensor)

  # convert logits to probs
  probs = torch.nn.functional.softmax(output, dim=1)

  # find index of max prob
  value, index = torch.max(probs, dim=1)

  if value < threshold:
    print("I don't know")

  print(list(vocab.keys())[index])

In [60]:
predict(model, "What is the largest planet in our solar system?")

jupiter
